# Train static hand signs

SOM course archive, organized during repository preparation. The MLP architecture is adapted from Kazuhito Takahashi and Nikita Kiselov. This notebook uses the shared `train.py` entry point. The supplied pretrained models remain unchanged.

Install `requirements-notebooks.txt` and launch Jupyter from the repository root.

In [ ]:
from pathlib import Path
from train import train


## Choose the dataset

The default uses the bundled course dataset. Set `dataset` to a different CSV to train on your recordings. Class IDs must match the label CSV. For new gestures, extend the label list and provide samples for every class.

In [ ]:
task = 'keypoint'
dataset = None  # Example: Path('data/recordings/keypoint.csv')
output = Path('artifacts') / task
epochs = 200
seed = 42


## Train and export

Exact duplicate rows are removed before a stratified 60% / 20% / 20% split. Validation selects the stopping point; the test split is used only for the final report. Exports go to `artifacts/`.

**Evaluation limit:** these datasets have no participant/session IDs. Neighbouring frames may still be similar across splits. Results do not establish performance on new people or recording sessions.

In [ ]:
report = train(task=task, dataset=dataset, output=output, epochs=epochs, seed=seed)
report


## Inspect errors

Rows are true classes; columns are predicted classes. This report belongs to the model trained above, not the archived 2024 model.

In [ ]:
import csv
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
with open(f'model/{task}_classifier/{task}_classifier_label.csv', encoding='utf-8-sig') as f:
    labels = [row[0] for row in csv.reader(f) if row]
display = ConfusionMatrixDisplay(report['confusion_matrix'], display_labels=labels)
# sklearn expects an array for plotting
import numpy as np
display.confusion_matrix = np.asarray(display.confusion_matrix)
display.plot(cmap='Blues', xticks_rotation=30)
plt.tight_layout()


## Use a new model

After checking its class order and results, copy the exported `.tflite` to the matching `model/` folder. Preserve the label order. Keep the original model as a backup. No automatic replacement occurs.